In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
df=pd.read_csv('plant_growth_data.csv')

In [4]:
df.head()

,Soil_Type,Sunlight_Hours,Water_Frequency,Fertilizer_Type,Temperature,Humidity,Growth_Milestone
0,loam,5.192294,bi-weekly,chemical,31.719602,61.591861,0
1,sandy,4.033133,weekly,organic,28.919484,52.422276,1
2,loam,8.892769,bi-weekly,none,23.179059,44.660539,0
3,loam,8.241144,bi-weekly,none,18.465886,46.433227,0
4,sandy,8.374043,bi-weekly,organic,18.128741,63.625923,0


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isnull().sum()

Soil_Type           0
Sunlight_Hours      0
Water_Frequency     0
Fertilizer_Type     0
Temperature         0
Humidity            0
Growth_Milestone    0
dtype: int64

In [7]:
df.nunique()

Soil_Type             3
Sunlight_Hours      190
Water_Frequency       3
Fertilizer_Type       3
Temperature         189
Humidity            191
Growth_Milestone      2
dtype: int64

In [8]:
df

,Soil_Type,Sunlight_Hours,Water_Frequency,Fertilizer_Type,Temperature,Humidity,Growth_Milestone
0,loam,5.192294,bi-weekly,chemical,31.719602,61.591861,0
1,sandy,4.033133,weekly,organic,28.919484,52.422276,1
2,loam,8.892769,bi-weekly,none,23.179059,44.660539,0
3,loam,8.241144,bi-weekly,none,18.465886,46.433227,0
4,sandy,8.374043,bi-weekly,organic,18.128741,63.625923,0
...,...,...,...,...,...,...,...
188,sandy,5.652000,daily,none,28.000000,70.200000,0
189,clay,7.528000,weekly,chemical,30.500000,60.100000,1
190,loam,4.934000,bi-weekly,none,24.500000,61.700000,0
191,sandy,8.273000,daily,organic,27.900000,69.500000,1


In [9]:
object_columns = df.select_dtypes(include=['object']).columns
print("Object type columns:")
print(object_columns)

numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns
print("\nNumerical type columns:")
print(numerical_columns)

Object type columns:
Index(['Soil_Type', 'Water_Frequency', 'Fertilizer_Type'], dtype='object')

Numerical type columns:
Index(['Sunlight_Hours', 'Temperature', 'Humidity', 'Growth_Milestone'], dtype='object')


In [10]:
def classify_features(df):
    categorical_features = []
    non_categorical_features = []
    discrete_features = []
    continuous_features = []

    for column in df.columns:
        if df[column].dtype == 'object':
            if df[column].nunique() < 30:
                categorical_features.append(column)
            else:
                non_categorical_features.append(column)
        elif df[column].dtype in ['int64', 'float64']: # here 64 is the 64 bit Operating System
            if df[column].nunique() < 30:
                discrete_features.append(column)
            else:
                continuous_features.append(column)

    return categorical_features, non_categorical_features, discrete_features, continuous_features

In [11]:
categorical, non_categorical, discrete, continuous = classify_features(df)

In [15]:
print("Categorical Features:", categorical)
print("Non-Categorical Features:", non_categorical)
print("Discrete Features:", discrete)
print("Continuous Features:", continuous)

Categorical Features: ['Soil_Type', 'Water_Frequency', 'Fertilizer_Type']
Non-Categorical Features: []
Discrete Features: ['Growth_Milestone']
Continuous Features: ['Sunlight_Hours', 'Temperature', 'Humidity']


In [ ]:
# Encode categorical features except target
feature_cols = ['Soil_Type', 'Water_Frequency', 'Fertilizer_Type']
df = pd.get_dummies(df, columns=feature_cols, drop_first=False)


In [17]:
df

,Sunlight_Hours,Temperature,Humidity,Growth_Milestone,Soil_Type_clay,Soil_Type_loam,Soil_Type_sandy,Water_Frequency_bi-weekly,Water_Frequency_daily,Water_Frequency_weekly,Fertilizer_Type_chemical,Fertilizer_Type_none,Fertilizer_Type_organic
0,5.192294,31.719602,61.591861,0,False,True,False,True,False,False,True,False,False
1,4.033133,28.919484,52.422276,1,False,False,True,False,False,True,False,False,True
2,8.892769,23.179059,44.660539,0,False,True,False,True,False,False,False,True,False
3,8.241144,18.465886,46.433227,0,False,True,False,True,False,False,False,True,False
4,8.374043,18.128741,63.625923,0,False,False,True,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,5.652000,28.000000,70.200000,0,False,False,True,False,True,False,False,True,False
189,7.528000,30.500000,60.100000,1,True,False,False,False,False,True,True,False,False
190,4.934000,24.500000,61.700000,0,False,True,False,True,False,False,False,True,False
191,8.273000,27.900000,69.500000,1,False,False,True,False,True,False,False,False,True


In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()
df[continuous]=scaler.fit_transform(df[continuous])

In [20]:
# Target encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['Growth_Milestone'])
X = df.drop('Growth_Milestone', axis=1)

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler

X = df.drop('Growth_Milestone', axis=1)
y = df['Growth_Milestone']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print("Before Oversampling:", y_train.value_counts())

Before Oversampling: Growth_Milestone
0    77
1    77
Name: count, dtype: int64


In [22]:
ros = RandomOverSampler(random_state=42)
#we apply sampling on only train data
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print("After Oversampling:", y_train_res.value_counts())

After Oversampling: Growth_Milestone
0    77
1    77
Name: count, dtype: int64


In [23]:
log_reg = LogisticRegression(max_iter=1000, solver='liblinear')
log_reg.fit(X_train_res, y_train_res)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [24]:
y_pred = log_reg.predict(X_test)

In [25]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.5128205128205128

Confusion Matrix:
 [[11  9]
 [10  9]]

Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.55      0.54        20
           1       0.50      0.47      0.49        19

    accuracy                           0.51        39
   macro avg       0.51      0.51      0.51        39
weighted avg       0.51      0.51      0.51        39



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Sunlight_Hours             193 non-null    float64
 1   Temperature                193 non-null    float64
 2   Humidity                   193 non-null    float64
 3   Growth_Milestone           193 non-null    int64  
 4   Soil_Type_clay             193 non-null    bool   
 5   Soil_Type_loam             193 non-null    bool   
 6   Soil_Type_sandy            193 non-null    bool   
 7   Water_Frequency_bi-weekly  193 non-null    bool   
 8   Water_Frequency_daily      193 non-null    bool   
 9   Water_Frequency_weekly     193 non-null    bool   
 10  Fertilizer_Type_chemical   193 non-null    bool   
 11  Fertilizer_Type_none       193 non-null    bool   
 12  Fertilizer_Type_organic    193 non-null    bool   
dtypes: bool(9), float64(3), int64(1)
memory usage: 7.9

In [29]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [30]:
logreg = LogisticRegression()

In [31]:
logreg.fit(X_train_resampled, y_train_resampled)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [32]:
y_pred = logreg.predict(X_test)

In [33]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.5128205128205128


In [34]:
report = classification_report(y_test, y_pred)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.55      0.54        20
           1       0.50      0.47      0.49        19

    accuracy                           0.51        39
   macro avg       0.51      0.51      0.51        39
weighted avg       0.51      0.51      0.51        39



In [35]:
from sklearn import svm
svc = svm.SVC()
svc.fit(X_train_resampled, y_train_resampled)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [36]:
y_pred = svc.predict(X_test)

In [37]:
report = classification_report(y_test, y_pred)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.50      0.53        20
           1       0.52      0.58      0.55        19

    accuracy                           0.54        39
   macro avg       0.54      0.54      0.54        39
weighted avg       0.54      0.54      0.54        39



In [38]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.5384615384615384


In [39]:
from sklearn.tree import DecisionTreeClassifier

# Create a Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=42)

# Train the classifier
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate the classifier
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.4358974358974359
Classification Report:
               precision    recall  f1-score   support

           0       0.44      0.35      0.39        20
           1       0.43      0.53      0.48        19

    accuracy                           0.44        39
   macro avg       0.44      0.44      0.43        39
weighted avg       0.44      0.44      0.43        39

